# Interpretable model V2 (v4 data): optimal tree and random forest

Same structure as `interpretable_model_V1_less_features.ipynb`, rebuilt on the **v4 master table** (`master_features_v4.csv`) with two models only: an **optimal classification tree** (OCT, Interpretable AI through Julia) and a **random forest** (RF). We start from **all 32 candidate features**, look at which ones matter, then drop the ones that don't help.

What changed from V1, because the v4 table changed the rules (see the v4 data dictionary):
- **Target:** `label` = **1 for plantation**, 0 for natural forest (V1 predicted forest). Every score below is the predicted probability of *plantation*.
- **Training rows:** `role == 'train'`: 1,700 MapBiomas points plus all 414 original Cropin QC polygons. The MapBiomas `lockbox` is sealed and not used.
- **Choosing settings:** there is no validation split any more, so settings are chosen by **spatially blocked 5-fold cross-validation** (CV): points are grouped by their spatial `block`, and a whole block is always in the same fold, so a model is never scored on neighbours of its training points. Every training point gets an **out-of-fold** score from a model that never saw its block. We judge on the **414 QC polygons only**, since those are Cropin's real alerts.
- **Test:** the **November Cropin QC set** (`role == 'test_nov'`), scored **once**, at the end, after the features, the models and the cutoff are fixed. Only the 153 unique points (`dup_first`).
- **Benchmark:** **SAG alone** (Cropin's one-season SAG, `nc_A_SAG`), used directly as a score.

Metrics:
- **AUC** = chance that a random plantation gets a higher plantation score than a random forest (0.5 = coin flip, 1 = perfect).
- **Cutoff:** a point is called plantation when its score is above a cutoff. The cutoff is chosen on the out-of-fold scores of the QC training polygons so that **at most 5% of QC forests are called plantation**, then applied unchanged to the November set.
- **Plantations caught** = share of plantations called plantation at that cutoff (the false alarms removed). **Forests called plantation** = share of real forests called plantation (deforestation that would be hidden).

In [ ]:
import os, subprocess, tempfile, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba, LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
warnings.filterwarnings('ignore', message='.*matmul.*')  # spurious NumPy/Accelerate warnings on macOS

In [ ]:
df = pd.read_csv('../Master Data/master_features_v4.csv')
X_all = list(df.loc[:, 'nc_B_SAG_SON':'lh_ever_open'].columns)  # the 32 columns with role "feature" in the v4 dictionary
mb_derived = ['nc_H_mapbiomas10m_plantation', 'lh_natural_run_years', 'lh_n_changes', 'lh_ever_open']  # built from MapBiomas maps
SAG = 'nc_A_SAG'  # benchmark: Cropin's SAG, Oct–Jan, one season
y = 'label'       # 1 = plantation, 0 = natural forest

train = df[df.role == 'train'].dropna(subset=X_all).reset_index(drop=True)  # 1 QC polygon has no radar data
test = df[(df.role == 'test_nov') & (df.dup_first == True)].reset_index(drop=True)  # unique November points only
is_qc = (train.set == 'qc').to_numpy()  # settings and cutoff are judged on these 413 rows

# Spatially blocked folds: every point of a block lands in the same fold
train['fold'] = -1
for k, (_, out) in enumerate(GroupKFold(n_splits=5, shuffle=True, random_state=0).split(train, groups=train.block)):
    train.loc[out, 'fold'] = k

print(len(X_all), 'candidate features')
pd.DataFrame({'rows': [len(train), is_qc.sum(), len(test)],
              'plantations': [train[y].sum(), train.loc[is_qc, y].sum(), test[y].sum()]},
             index=['train (all)', 'train (QC only)', 'test (November QC, unique)'])

In [ ]:
def cutoff(p_forest):
    """Smallest cutoff that calls at most 5% of these forests plantation (rule: plantation if score > cutoff)."""
    return np.quantile(p_forest, 0.95, method='higher')

oof, test_p = {}, {}  # out-of-fold scores (train), test scores (kept unseen until section 7)
cv_cuts, cuts, cut_scores = {}, {}, {}  # cutoff for the CV numbers, cutoff applied to the November set, scores it was placed on
qc_forest = is_qc & (train[y] == 0).to_numpy()

def register(name, p_oof, p_test, p_cut=None):
    """Store a model's scores and fix its cutoffs on the QC forests.
    p_cut: training scores used to place the November cutoff (default: the out-of-fold scores)."""
    oof[name], test_p[name] = np.asarray(p_oof, float), np.asarray(p_test, float)
    cv_cuts[name] = cutoff(oof[name][qc_forest])
    cut_scores[name] = oof[name] if p_cut is None else np.asarray(p_cut, float)
    cuts[name] = cutoff(cut_scores[name][qc_forest])

def cv_row(name):
    """Out-of-fold performance on the QC training polygons (and AUC on MapBiomas points, for context)."""
    p, c, yq = oof[name][is_qc], cv_cuts[name], train.loc[is_qc, y].to_numpy()
    return {'model': name,
            'CV AUC (QC)': roc_auc_score(yq, p),
            'plantations caught (QC)': (p[yq == 1] > c).mean(),
            'forests called plantation (QC)': (p[yq == 0] > c).mean(),
            'CV AUC (MapBiomas)': roc_auc_score(train.loc[~is_qc, y], oof[name][~is_qc])}

def rf_oof(cols, **params):
    """Out-of-fold RF scores: each fold is scored by a forest fit on the other four."""
    p = np.zeros(len(train))
    for k in range(5):
        fit, out = train.fold != k, train.fold == k
        m = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1, **params).fit(train.loc[fit, cols], train.loc[fit, y])
        p[out.to_numpy()] = m.predict_proba(train.loc[out, cols])[:, 1]
    return p

## 0. Correlated features

Two features that move together carry nearly the same information. A tree then picks between them almost at random, and a random forest splits the credit between them, so both look less important than they are. We measure this with the **Spearman correlation** (from −1 to +1): +1 = the two features rank the points in exactly the same order, −1 = in exactly the opposite order, 0 = no relation. It is computed on the training points only.

Unlike V1, nothing is dropped yet: both models start with all 32 features. Features are ordered so that correlated ones sit next to each other, and numbers are shown where |correlation| ≥ 0.8. These pairs are revisited in section 4.

In [ ]:
def corr_map(cols, title):
    rho = train[cols].corr(method='spearman')
    order = leaves_list(linkage(squareform(1 - rho.abs(), checks=False), 'average'))  # correlated features side by side
    rho = rho.iloc[order, order]
    low = rho.iloc[1:, :-1]  # lower triangle without the diagonal
    cmap = LinearSegmentedColormap.from_list('div', ['#3B6EA8', '#F2F2F2', '#750014'])  # blue = negative, burgundy = positive
    fig, ax = plt.subplots(figsize=(13, 11))
    im = ax.imshow(low.where(np.tril(np.ones(low.shape, bool))), cmap=cmap, vmin=-1, vmax=1)
    for i in range(len(low)):
        for j in range(i + 1):
            if abs(low.iat[i, j]) >= 0.8:
                ax.text(j, i, f'{low.iat[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white')
    ax.set_xticks(range(low.shape[1]), low.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(low.shape[0]), low.index, fontsize=8)
    ax.spines[:].set_visible(False)
    fig.colorbar(im, ax=ax, shrink=0.6, label='Spearman correlation (training points)')
    ax.set_title(title, loc='left')
    fig.tight_layout()
    plt.show()

def top_pairs(cols, n=12):
    C = train[cols].corr(method='spearman')
    iu = np.triu_indices(len(cols), 1)
    pairs = pd.DataFrame({'feature A': C.index[iu[0]], 'feature B': C.columns[iu[1]], 'Spearman': C.values[iu]})
    return pairs.reindex(pairs.Spearman.abs().sort_values(ascending=False).index).head(n).reset_index(drop=True)

corr_map(X_all, f'Correlations between all {len(X_all)} candidate features')
top_pairs(X_all).round(3)

## 1. Benchmark: SAG alone

Cropin's SAG (Oct–Jan, one season) used directly as a plantation score: plantations have a higher SAG than natural forest in this data. Nothing is fitted except the cutoff, which is set on the 413 QC training polygons like every other model's.

In [ ]:
register('SAG alone', train[SAG], test[SAG])
pd.DataFrame([cv_row('SAG alone')]).set_index('model').round(3)

## 2. Random forest, all 32 features

A **random forest** averages 500 decision trees, each grown on a random resample of the training points and allowed to look at only a random subset of features at each split. It is not readable like a single tree, but it is a strong, low-maintenance baseline, and section 3 shows which features drive it.

Two settings are chosen by out-of-fold AUC on the QC polygons: `max_features` (share of features each split may look at) and `min_samples_leaf` (smallest leaf, larger = smoother scores).

In [ ]:
rf_grid = [dict(max_features=mf, min_samples_leaf=msl) for mf in ['sqrt', 0.2, 0.33] for msl in [1, 5, 20]]

def tune_rf(cols):
    """Out-of-fold scores for each setting; keep the best QC AUC."""
    runs = [(g, rf_oof(cols, **g)) for g in rf_grid]
    aucs = [roc_auc_score(train.loc[is_qc, y], p[is_qc]) for _, p in runs]
    best = int(np.argmax(aucs))
    print('CV AUC (QC) by setting:', {f"{g['max_features']}/{g['min_samples_leaf']}": round(a, 3) for (g, _), a in zip(runs, aucs)})
    return runs[best]

def fit_rf(cols, params):
    return RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1, **params).fit(train[cols], train[y])

rf_params_all, p = tune_rf(X_all)
rf_all = fit_rf(X_all, rf_params_all)  # refit on all training rows; its test scores stay unseen until section 7
register('RF, all features', p, rf_all.predict_proba(test[X_all])[:, 1])
print('Chosen:', rf_params_all)
pd.DataFrame([cv_row(m) for m in oof]).set_index('model').round(3)

## 3. Optimal classification tree, all 32 features (Interpretable AI, run through Julia)

CART grows a tree greedily, one split at a time. An **optimal tree** instead searches for the best tree of a given depth as a whole (Interpretable AI's Optimal Trees, the MIT method). As in V1, the notebook writes the data to a temporary folder outside the repo and runs a Julia helper with the IAI build of Julia (`julia +1.11.6/IAI/3.2.2`). The helper here, `helper_train_iai_cv.jl`, also runs the same 5 spatial folds, so the tree gets out-of-fold scores comparable to the forest's.

- **Depth 1–5**, chosen by out-of-fold AUC on the QC polygons (capped at 5 so the tree stays readable). For each fit, IAI tunes its own complexity penalty (`cp`) on an internal split of that fit's training rows. The leaf minimum is 20 points.
- A tree only predicts a class, so each point's score is its leaf's **share of plantations among training points**.
- **Cutoff for a tree:** the out-of-fold scores come from 5 different fold trees, whose leaves hold different shares than the final tree's, so a cutoff placed between fold-tree leaf values doesn't carry over to the final tree (in a first run it called every November point forest). For the November set, the tree's cutoff is therefore placed on the **final tree's own leaves**: the leaves are ranked by plantation share, and leaves are called plantation, from the most plantation-like down, as long as at most 5% of the QC training forests fall in them. This uses in-sample scores, so it can be slightly optimistic; the CV numbers still use the out-of-fold cutoff. The table under each tree shows, leaf by leaf, how many QC training polygons it holds and which leaves are called plantation.

Takes about 2–4 minutes, including up to a minute for Julia to start.

In [ ]:
RUN_IAI = True  # set False to skip (needs juliaup with the IAI channel)
IAI_JULIA = ['julia', '+1.11.6/IAI/3.2.2']  # juliaup channel with IAI preloaded (plain `julia` has no IAI)

def run_iai(cols, name):
    """Run the CV + refit in Julia, pick the depth on QC out-of-fold AUC, register the model. Returns (depth, folder)."""
    iai_dir = tempfile.mkdtemp(prefix='iai_')  # outside the repo: these CSVs hold one row per point
    train[cols].assign(y=train[y], fold=train.fold).to_csv(os.path.join(iai_dir, 'train.csv'), index=False)
    test[cols].assign(y=test[y]).to_csv(os.path.join(iai_dir, 'test.csv'), index=False)
    run = subprocess.run(IAI_JULIA + [os.path.abspath('helper_train_iai_cv.jl'), iai_dir, '5', '20'],
                         capture_output=True, text=True, encoding='utf-8', errors='replace')
    assert run.returncode == 0, run.stderr[-1500:]
    print(run.stdout)
    p_oof, p_train, p_test = (pd.read_csv(os.path.join(iai_dir, f)) for f in ['oof.csv', 'p_train.csv', 'p_test.csv'])
    aucs = {int(c.split('_')[1]): roc_auc_score(train.loc[is_qc, y], p_oof.loc[is_qc, c]) for c in p_oof}
    print('CV AUC (QC) by depth:', {d: round(a, 3) for d, a in aucs.items()})
    depth = max(aucs, key=aucs.get)
    print(f'Chosen: depth {depth} (tree drawn below)')
    register(f'{name} (depth {depth})', p_oof[f'depth_{depth}'], p_test[f'depth_{depth}'], p_cut=p_train[f'depth_{depth}'])
    return depth, iai_dir

oct_all = None
if RUN_IAI:
    try:
        oct_all = run_iai(X_all, 'OCT, all features')
    except Exception as e:
        print('IAI optimal tree skipped:', type(e).__name__, str(e)[:1500])
pd.DataFrame([cv_row(m) for m in oof]).set_index('model').round(3)

In [ ]:
# Same format as V1: burgundy = plantation, blue = forest; darker = purer node.
# Each box: split rule (points with feature < threshold go left, "True"), share of training points, class mix.
def plot_iai_tree(nodes, ax, fontsize=8):
    nodes = nodes.set_index('node')
    pos, n_leaves = {}, [0]
    def place(n):  # leaves left to right, each parent centred over its two children
        r = nodes.loc[n]
        if r.is_leaf:
            pos[n] = n_leaves[0]; n_leaves[0] += 1
        else:
            place(r.lower); place(r.upper); pos[n] = (pos[r.lower] + pos[r.upper]) / 2
    place(1)
    xy = {n: (pos[n], -nodes.loc[n, 'depth']) for n in nodes.index}
    for n, r in nodes[~nodes.is_leaf].iterrows():  # edges, drawn under the boxes
        for child, word in [(r.lower, 'True'), (r.upper, 'False')]:
            ax.plot(*zip(xy[n], xy[child]), color='grey', lw=1, zorder=1)
            if n == 1:
                ax.text(*np.mean([xy[n], xy[child]], axis=0), word, ha='center', va='center', fontsize=fontsize,
                        bbox=dict(boxstyle='round', fc='white', ec='none'), zorder=2)
    n_total = nodes.loc[1, 'n_train']
    for n, r in nodes.iterrows():
        f = r.p1  # share of plantations in the node
        a = min(1.2 * abs(f - 0.5), 1)
        fill = tuple(a * np.array(to_rgba('#750014' if f >= 0.5 else '#3B6EA8')[:3]) + (1 - a))
        text = ('' if r.is_leaf else f'{r.feature} < {r.threshold:.2f}\n') + \
               f'samples = {100 * r.n_train / n_total:.1f}%\nvalue = [{1 - f:.2f}, {f:.2f}]\nclass = {"plantation" if f >= 0.5 else "forest"}'
        ax.text(*xy[n], text, ha='center', va='center', fontsize=fontsize, zorder=3,
                bbox=dict(boxstyle='round', ec='grey', fc=fill))
    ax.set_xlim(-0.6, n_leaves[0] - 0.4); ax.set_ylim(-nodes.depth.max() - 0.5, 0.5); ax.axis('off')

def draw_oct(fit, title):
    if fit is None:
        return
    depth, iai_dir = fit
    nodes = pd.read_csv(os.path.join(iai_dir, f'tree_depth_{depth}_nodes.csv'), keep_default_na=False, na_values=['NaN'])
    fig, ax = plt.subplots(figsize=(30, 12))
    plot_iai_tree(nodes, ax)
    ax.set_title(title + '   (value = [forest, plantation] share)', loc='left', fontsize=14)
    plt.show()

def leaf_table(name):
    """QC training polygons in each leaf of the final tree, most plantation-like leaf first."""
    d = pd.DataFrame({'leaf plantation share': cut_scores[name][is_qc].round(3), 'label': train.loc[is_qc, y].to_numpy()})
    t = d.groupby('leaf plantation share').label.agg(**{'QC plantations': 'sum', 'QC forests': lambda s: (s == 0).sum()})
    t = t.sort_index(ascending=False)
    t['cumulative share of QC forests'] = (t['QC forests'].cumsum() / qc_forest.sum()).round(3)
    t['called plantation in November'] = t.index > cuts[name]
    return t

draw_oct(oct_all, 'Optimal tree, all features, refit on all training rows')
if oct_all is not None:
    display(leaf_table(next(m for m in oof if m.startswith('OCT, all'))))

## 3b. Feature relative importance, all 32 features

Three views, each rescaled so that a model's importances add up to 100%:
- **RF, permutation (main view):** in each CV fold, shuffle one feature among the held-out QC polygons and measure how much the AUC drops (5 shuffles per feature and fold, averaged). A feature the model doesn't need loses nothing. Because it is measured on held-out Cropin polygons, this is the most honest view of what helps *at Cropin's alerts*. Negative values (shuffling helped) are shown as 0 in the shares. Caveat: two correlated features share the credit, so both can look weak.
- **RF, impurity:** how much each feature's splits purify the training data, summed over all trees. Fast, but measured on training data and biased towards continuous features.
- **OCT:** Interpretable AI's importance for the refit tree (how much its splits on each feature improve the tree). Features the tree doesn't use get 0.

In [ ]:
def rf_permutation(cols, params, n_repeats=5, seed=0):
    """Mean drop in held-out QC AUC when one feature is shuffled, across the 5 spatial folds."""
    rng = np.random.default_rng(seed)
    drops = pd.DataFrame(0.0, index=range(5), columns=cols)
    for k in range(5):
        fit, out = train.fold != k, (train.fold == k).to_numpy() & is_qc
        m = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1, **params).fit(train.loc[fit, cols], train.loc[fit, y])
        Xo, yo = train.loc[out, cols].reset_index(drop=True), train.loc[out, y]
        base = roc_auc_score(yo, m.predict_proba(Xo)[:, 1])
        for c in cols:
            shuffled = []
            for _ in range(n_repeats):
                Xs = Xo.copy(); Xs[c] = rng.permutation(Xs[c].to_numpy())
                shuffled.append(roc_auc_score(yo, m.predict_proba(Xs)[:, 1]))
            drops.loc[k, c] = base - np.mean(shuffled)
    return drops.mean()

def importance_table(cols, rf_model, rf_params, oct_fit):
    t = pd.DataFrame(index=cols)
    t['RF permutation: AUC drop'] = rf_permutation(cols, rf_params)
    share = lambda s: 100 * s.clip(lower=0) / s.clip(lower=0).sum()
    t['RF permutation (%)'] = share(t['RF permutation: AUC drop'])
    t['RF impurity (%)'] = share(pd.Series(rf_model.feature_importances_, index=cols))
    if oct_fit is not None:
        imp = pd.read_csv(os.path.join(oct_fit[1], f'importance_depth_{oct_fit[0]}.csv')).set_index('Feature').Importance
        t['OCT (%)'] = share(imp.reindex(cols).fillna(0))
    return t.sort_values('RF permutation (%)', ascending=False)

def plot_importance(t, title):
    views = [('RF permutation (%)', '#750014'), ('RF impurity (%)', '#B3B3B3'), ('OCT (%)', '#3B6EA8')]
    views = [v for v in views if v[0] in t]
    fig, ax = plt.subplots(figsize=(10, 0.32 * len(t) + 1.5))
    ypos, h = np.arange(len(t)), 0.8 / len(views)
    for i, (col, color) in enumerate(views):
        ax.barh(ypos + (i - (len(views) - 1) / 2) * h, t[col], height=h, color=color, label=col.replace(' (%)', ''))
    ax.set_yticks(ypos, t.index, fontsize=8); ax.invert_yaxis()
    ax.set_xlabel('Relative importance (% of the model\'s total)')
    ax.legend(frameon=False, loc='lower right'); ax.spines[['top', 'right']].set_visible(False)
    ax.set_title(title, loc='left')
    fig.tight_layout(); plt.show()

imp_all = importance_table(X_all, rf_all, rf_params_all, oct_all)
plot_importance(imp_all, f'Feature relative importance, all {len(X_all)} features (sorted by RF permutation)')
imp_all.round(3)

## 4. Dropping features

Three rules, applied in this order, using training data only (the November set is still unseen):
1. **MapBiomas-derived features** (`nc_H_mapbiomas10m_plantation` and the three `lh_` features) are dropped: they are built from MapBiomas maps, and 80% of the training points are MapBiomas points, so on those points they partly encode the label (known issue 1 in the v4 dictionary).
2. **Correlated pairs** (|Spearman| ≥ 0.8, section 0): of each pair, the one with the lower RF permutation importance is dropped, strongest pair first.
3. **Features that don't help at Cropin's alerts:** shuffling them changes the held-out QC AUC by less than 0.001 (0.1 AUC points) in the RF permutation view.

Both models are then retuned on the reduced set exactly as before (same folds, same grids), and section 6 compares CV performance before and after. Fewer features matter here: every feature is one more GEE extraction to maintain, and one more thing to explain to Cropin.

In [ ]:
perm = imp_all['RF permutation: AUC drop']
dropped = {c: 'MapBiomas-derived' for c in mb_derived}
kept = [c for c in X_all if c not in dropped]
for _, r in top_pairs(kept, n=len(kept) ** 2).iterrows():
    a, b = r['feature A'], r['feature B']
    if abs(r.Spearman) < 0.8:
        break
    if a in dropped or b in dropped:
        continue
    weaker = a if perm[a] < perm[b] else b
    dropped[weaker] = f'correlated with {b if weaker == a else a} (rho {r.Spearman:.2f})'
for c in X_all:
    if c not in dropped and perm[c] < 0.001:
        dropped[c] = f'RF permutation AUC drop {perm[c]:.4f} < 0.001'
X_red = [c for c in X_all if c not in dropped]

print(f'{len(X_red)} features kept:', X_red)
pd.DataFrame({'reason': dropped}).rename_axis('dropped feature')

In [ ]:
corr_map(X_red, f'Correlations between the {len(X_red)} features kept')
top_pairs(X_red, n=8).round(3)

## 5. Random forest and optimal tree on the reduced feature set

In [ ]:
rf_params_red, p = tune_rf(X_red)
rf_red = fit_rf(X_red, rf_params_red)
register('RF, reduced features', p, rf_red.predict_proba(test[X_red])[:, 1])
print('Chosen:', rf_params_red)

oct_red = None
if RUN_IAI:
    try:
        oct_red = run_iai(X_red, 'OCT, reduced features')
    except Exception as e:
        print('IAI optimal tree skipped:', type(e).__name__, str(e)[:1500])

In [ ]:
draw_oct(oct_red, 'Optimal tree, reduced features, refit on all training rows')
if oct_red is not None:
    display(leaf_table(next(m for m in oof if m.startswith('OCT, reduced'))))

### Feature relative importance, reduced feature set

Same three views as in section 3b, for the models refit on the reduced set. With fewer correlated features left, the credit is shared out less and the ranking is easier to read.

In [ ]:
imp_red = importance_table(X_red, rf_red, rf_params_red, oct_red)
plot_importance(imp_red, f'Feature relative importance, reduced set ({len(X_red)} features)')
imp_red.round(3)

## 6. Comparison in cross-validation (training data only)

Out-of-fold scores on the 413 QC training polygons: each polygon is scored by a model that never saw its spatial block. The cutoff is set on these same scores (at most 5% of QC forests called plantation), so `forests called plantation (QC)` is ≤ 5% by construction, and `plantations caught (QC)` is the number to compare. The MapBiomas AUC is for context only (history features look much better on MapBiomas points than at Cropin's alerts).

**AUC gain over SAG** = model AUC − SAG-alone AUC on the same QC polygons.

In [ ]:
cv = pd.DataFrame([cv_row(m) for m in oof]).set_index('model')
cv.insert(1, 'AUC gain over SAG (QC)', cv['CV AUC (QC)'] - cv.loc['SAG alone', 'CV AUC (QC)'])
cv['features'] = [1 if m == 'SAG alone' else len(X_all) if 'all' in m else len(X_red) for m in cv.index]
cv.round(3)

### ROC curves (cross-validation, QC training polygons)

An **ROC curve** shows every possible cutoff at once. Moving along it from left to right, the cutoff gets looser: more plantations are caught (up), but more real forest is called plantation too (right). A curve hugging the top-left corner is better; the grey diagonal is a coin flip. The dotted line marks 5% of forests called plantation, and the dot on each curve is the chosen cutoff. Trees give only one score per leaf, so their curves are a few straight segments.

In [ ]:
styles = {'SAG': ('#8C8C8C', '-'), 'RF, all': ('#750014', '--'), 'RF, reduced': ('#750014', '-'),
          'OCT, all': ('#3B6EA8', '--'), 'OCT, reduced': ('#3B6EA8', '-')}

def roc_panel(ax, scores, labels, title, cut):
    labels = np.asarray(labels)
    for name, p in scores.items():
        color, ls = next(v for k, v in styles.items() if name.startswith(k))
        fpr, tpr, _ = roc_curve(labels, p)  # x = forests called plantation, y = plantations caught
        ax.plot(fpr, tpr, color=color, ls=ls, lw=2, label=f'{name}  (AUC {roc_auc_score(labels, p):.3f})')
        flagged = p > cut[name]
        ax.plot(flagged[labels == 0].mean(), flagged[labels == 1].mean(), 'o', color=color, ms=7, mec='white', zorder=5)
    ax.plot([0, 1], [0, 1], color='#BDBDBD', lw=1, zorder=0)
    ax.axvline(0.05, color='#6E6E6E', ls=':', lw=1)
    ax.text(0.06, 0.03, '5% of forests called plantation', color='#6E6E6E', fontsize=9)
    ax.set(xlim=(0, 1), ylim=(0, 1.01), xlabel='Forests called plantation (share of real forests)', ylabel='Plantations caught (share)')
    ax.set_title(title, loc='left')
    ax.legend(loc='lower right', fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
roc_panel(ax, {m: p[is_qc] for m, p in oof.items()}, train.loc[is_qc, y], 'Cross-validation, 413 QC training polygons (dot = cutoff)', cv_cuts)
fig.tight_layout(); plt.show()

## 7. November QC test set (scored once)

Everything above was fixed on training data. Here the November set is scored **once**, with each model's cutoff applied unchanged, following the rules agreed on 24 Nov:
- **Three cuts:** all 153 unique points; without the 42 polygons within 50 m of an original QC polygon (`overlap_train_50m`, partly in-sample since those are now training); and the 6 `label_conflict` polygons (new forest within 50 m of an old plantation) on their own.
- **All four outcomes**, with **95% confidence intervals** (CI: the range the true value plausibly lies in, given only 153 points). CIs come from a **spatial block bootstrap**: we redraw the November blocks with replacement 2,000 times and recompute each number.

Caveats: the November set is small (92 forests, 69 plantations), covers only clearings inside Cropin's forest layer, and its `nc_hansen_tc2000` was sampled at 10 m rather than 30 m (known issue 5).

In [ ]:
def outcomes(p, labels, cut):
    flagged, labels = p > cut, np.asarray(labels)
    pl, fo = labels == 1, labels == 0
    return {'AUC': roc_auc_score(labels, p) if pl.any() and fo.any() else np.nan,
            'plantations caught': flagged[pl].mean() if pl.any() else np.nan,
            'plantations missed': (~flagged[pl]).mean() if pl.any() else np.nan,
            'forests kept': (~flagged[fo]).mean() if fo.any() else np.nan,
            'forests called plantation': flagged[fo].mean() if fo.any() else np.nan}

def block_bootstrap(rows, name, n_boot=2000, seed=0):
    """Point estimates and 95% CIs, resampling spatial blocks with replacement."""
    d = test[rows]
    est = outcomes(test_p[name][rows], d[y], cuts[name])
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(d.block.to_numpy() == b) for b in d.block.unique()]
    p, lab = test_p[name][rows], d[y].to_numpy()
    boots = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        boots.append(outcomes(p[idx], lab[idx], cuts[name]))
    boots = pd.DataFrame(boots)
    return {k: f'{v:.2f} [{boots[k].quantile(0.025):.2f}, {boots[k].quantile(0.975):.2f}]' if pd.notna(v) else '–'
            for k, v in est.items()}

subsets = {'all 153 unique points': np.ones(len(test), bool),
           'without 42 overlap_train_50m': (test.overlap_train_50m != 1).to_numpy(),
           '6 label_conflict': (test.label_conflict == 1).to_numpy()}
for s, rows in subsets.items():
    print(f'{s}: {rows.sum()} points ({test.loc[rows, y].sum()} plantations, {(test.loc[rows, y] == 0).sum()} forests)')

In [ ]:
res = {s: pd.DataFrame({m: block_bootstrap(rows, m) for m in oof}).T for s, rows in list(subsets.items())[:2]}
print('All 153 unique November points: estimate [95% CI]')
res['all 153 unique points']

In [ ]:
print('Without the 42 polygons within 50 m of a training QC polygon: estimate [95% CI]')
res['without 42 overlap_train_50m']

In [ ]:
# 6 polygons only: counts, no CI (AUC needs both classes)
lc = subsets['6 label_conflict']
pd.DataFrame({m: {'forests (of {})'.format((test.loc[lc, y] == 0).sum()) + ' called plantation': int(((test_p[m][lc] > cuts[m]) & (test.loc[lc, y] == 0).to_numpy()).sum()),
                  'plantations (of {})'.format(test.loc[lc, y].sum()) + ' caught': int(((test_p[m][lc] > cuts[m]) & (test.loc[lc, y] == 1).to_numpy()).sum())}
              for m in oof}).T

### ROC curves on the November set

Same reading as in section 6. The dot is where the cutoff fixed in cross-validation actually lands on new data: if it sits right of the dotted line, that model calls more than 5% of November forests plantation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)
for ax, (s, rows) in zip(axes, list(subsets.items())[:2]):
    roc_panel(ax, {m: p[rows] for m, p in test_p.items()}, test.loc[rows, y], f'November QC: {s}', cuts)
fig.suptitle('ROC curves on the November test set (dot = cutoff fixed in cross-validation)', x=0.01, ha='left')
fig.tight_layout(); plt.show()

## 8. Summary

_Filled in after the run: see the summary cell below._